# Percolation Summarization: TF/TF-IDF vs Fixed-Ratio on CNN/DM

This notebook demonstrates **percolation-threshold extractive summarization**.

For each document, a word co-occurrence graph is built over all sentences.
Sentences are greedily added (highest-scoring first) to a summary graph;
the **percolation threshold k\*** is the number of sentences needed for the summary
graph's largest connected component (GCC) to reach fraction θ of the full document GCC.
This is compared against fixed-ratio baselines (10%, 20%, 30% of sentences).

**Scorers:** TF (term frequency) and TF-IDF.
**Metrics:** ROUGE-1/2/L F/R/P.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru and rouge_score are NOT pre-installed on Colab
_pip('loguru==0.7.3', 'rouge-score==0.1.2')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1',
         'matplotlib==3.10.0', 'networkx==3.6.1', 'nltk==3.9.1')

In [ ]:
import gc
import json
import math
import os
import string
import sys
from collections import Counter
from typing import Any

import networkx as nx
import nltk
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression

# Download NLTK resources if needed
for res_id, res_type in [("punkt_tab", "tokenizers"), ("punkt", "tokenizers"), ("stopwords", "corpora")]:
    try:
        nltk.data.find(f"{res_type}/{res_id}")
    except LookupError:
        nltk.download(res_id, quiet=True)

from nltk.corpus import stopwords as _sw
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer as rs

STOPWORDS = set(_sw.words("english"))
PUNCT_TABLE = str.maketrans("", "", string.punctuation)
ROUGE_SCORER = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
print("Imports ready.")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6a9498-vocabulary-percolation-threshold-for-sel/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Loaded corpora:", [ds["dataset"] for ds in data["datasets"]])
cnndm_examples = data["datasets"][0]["examples"]
print(f"CNN/DM examples: {len(cnndm_examples)}")

## Config

Tunable parameters. Set to minimum values for quick demo; increase for full experiment.

In [ ]:
# Percolation thresholds (GCC fraction at which k* is recorded)
THETAS = [0.6, 0.7, 0.8, 0.9]

# Fixed-ratio baselines
FIXED_RATIOS = [0.10, 0.20, 0.30]

# Cap very long Multi-News docs
MAX_SENTENCES = 100

# Number of CNN/DM documents to process (original: 2000)
MAX_CNNDM = 3  # minimum for demo; original was 2000

## Helper Functions

Text preprocessing (sentence tokenization, content word extraction) and graph construction.

In [ ]:
def tokenize_sentences(text: str) -> list[str]:
    sents = sent_tokenize(text.strip())
    return [s.strip() for s in sents if s.strip() and len(s.split()) >= 3]


def get_content_words(sentence: str) -> list[str]:
    tokens = sentence.lower().translate(PUNCT_TABLE).split()
    return [w for w in tokens if w not in STOPWORDS and len(w) > 2]


def build_vocab_graph(cw_per_sent: list[list[str]]) -> nx.Graph:
    G: nx.Graph = nx.Graph()
    for cws in cw_per_sent:
        unique = list(set(cws))
        G.add_nodes_from(unique)
        for i in range(len(unique)):
            for j in range(i + 1, len(unique)):
                u, v = unique[i], unique[j]
                if G.has_edge(u, v):
                    G[u][v]["weight"] += 1
                else:
                    G.add_edge(u, v, weight=1)
    return G


def gcc_size(G: nx.Graph) -> int:
    if len(G.nodes) == 0:
        return 0
    return len(max(nx.connected_components(G), key=len))

## Scoring Functions

Two sentence scoring methods: TF (term frequency) and TF-IDF.

In [ ]:
def tf_scores(cw_per_sent: list[list[str]]) -> list[float]:
    freq: Counter = Counter()
    for cws in cw_per_sent:
        freq.update(cws)
    return [
        sum(freq[w] for w in cws) / len(cws) if cws else 0.0
        for cws in cw_per_sent
    ]


def tfidf_scores(sentences: list[str]) -> list[float]:
    if len(sentences) < 2:
        return [1.0] * len(sentences)
    try:
        vec = TfidfVectorizer(stop_words="english", min_df=1)
        X = vec.fit_transform(sentences)
        return list(np.asarray(X.sum(axis=1)).flatten())
    except Exception:
        return [0.0] * len(sentences)

## Percolation Pipeline

Core algorithm: greedily add sentences (by score) to a summary graph and record
after how many sentences (k*) the summary GCC reaches fraction θ of the full document GCC.

In [ ]:
def percolation_summary(
    cw_per_sent: list[list[str]],
    sent_scores: list[float],
    full_gcc: int,
) -> dict[str, Any]:
    n = len(sent_scores)
    order = sorted(range(n), key=lambda i: sent_scores[i], reverse=True)

    results: dict[float, int | None] = {t: None for t in THETAS}
    G_summary: nx.Graph = nx.Graph()
    gcc_curve: list[float] = []

    for step, idx in enumerate(order):
        cws = list(set(cw_per_sent[idx]))
        G_summary.add_nodes_from(cws)
        for i in range(len(cws)):
            for j in range(i + 1, len(cws)):
                u, v = cws[i], cws[j]
                if G_summary.has_edge(u, v):
                    G_summary[u][v]["weight"] += 1
                else:
                    G_summary.add_edge(u, v, weight=1)
        cur_gcc = gcc_size(G_summary)
        ratio = cur_gcc / full_gcc if full_gcc > 0 else 0.0
        gcc_curve.append(ratio)
        for t in THETAS:
            if results[t] is None and ratio >= t:
                results[t] = step + 1

    k_star = {}
    ceiling_hit = {}
    for t in THETAS:
        if results[t] is None:
            ceiling_hit[t] = True
            k_star[t] = n
        else:
            ceiling_hit[t] = False
            k_star[t] = results[t]

    compression = {t: k_star[t] / n for t in THETAS}
    return {
        "k_star": k_star,
        "compression": compression,
        "ceiling_hit": ceiling_hit,
        "gcc_curve": gcc_curve,
    }

## ROUGE Evaluation and Fixed-Ratio Baseline

ROUGE-1/2/L scores against the reference summary.

In [ ]:
def evaluate_summary(selected: list[int], sentences: list[str], reference: str) -> dict[str, float]:
    if not selected:
        return {k: 0.0 for k in ["rouge1_f","rouge1_r","rouge1_p","rouge2_f","rouge2_r","rougeL_f","rougeL_r"]}
    text = " ".join(sentences[i] for i in selected)
    s = ROUGE_SCORER.score(reference, text)
    return {
        "rouge1_f": s["rouge1"].fmeasure,
        "rouge1_r": s["rouge1"].recall,
        "rouge1_p": s["rouge1"].precision,
        "rouge2_f": s["rouge2"].fmeasure,
        "rouge2_r": s["rouge2"].recall,
        "rougeL_f": s["rougeL"].fmeasure,
        "rougeL_r": s["rougeL"].recall,
    }


def fixed_summary(n: int, scores: list[float], ratio: float) -> list[int]:
    k = max(1, round(n * ratio))
    top_k = sorted(range(n), key=lambda i: scores[i], reverse=True)[:k]
    return sorted(top_k)

## Per-Document Processing

Builds the full vocab graph, computes both scorers, runs percolation and fixed-ratio
summarization, and evaluates ROUGE for all variants.

In [ ]:
def process_document(article_text: str, reference_text: str, doc_id: str, corpus_name: str) -> dict | None:
    sentences = tokenize_sentences(article_text)
    if len(sentences) < 3:
        return None
    if len(sentences) > MAX_SENTENCES:
        sentences = sentences[:MAX_SENTENCES]

    cw_per_sent = [get_content_words(s) for s in sentences]
    full_G = build_vocab_graph(cw_per_sent)
    full_gcc = gcc_size(full_G)

    deg_vals = [d for _, d in full_G.degree()]
    avg_degree = float(np.mean(deg_vals)) if deg_vals else 0.0
    clustering = float(nx.average_clustering(full_G)) if full_G.nodes else 0.0
    density = float(nx.density(full_G))

    tf_s = tf_scores(cw_per_sent)
    tfidf_s = tfidf_scores(sentences)

    record: dict[str, Any] = {
        "doc_id": doc_id,
        "corpus": corpus_name,
        "n_sentences": len(sentences),
        "full_gcc": full_gcc,
        "avg_degree": round(avg_degree, 4),
        "clustering": round(clustering, 4),
        "density": round(density, 6),
    }

    for scorer_name, scores in [("tf", tf_s), ("tfidf", tfidf_s)]:
        perc = percolation_summary(cw_per_sent, scores, full_gcc)

        for theta in THETAS:
            k_star = perc["k_star"][theta]
            selected = sorted(
                sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k_star]
            )
            variant = f"{scorer_name}_theta{int(theta*10)}"
            rouge = evaluate_summary(selected, sentences, reference_text)
            record[f"{variant}_k_star"] = k_star
            record[f"{variant}_compression_ratio"] = round(perc["compression"][theta], 4)
            record[f"{variant}_ceiling_hit"] = perc["ceiling_hit"][theta]
            record[f"{variant}_rouge1_f"] = round(rouge["rouge1_f"], 5)
            record[f"{variant}_rouge2_f"] = round(rouge["rouge2_f"], 5)
            record[f"{variant}_rougeL_f"] = round(rouge["rougeL_f"], 5)

        # GCC decile growth curve
        curve = perc["gcc_curve"]
        nc = len(curve)
        deciles = []
        for frac in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
            idx = max(0, min(int(frac * nc) - 1, nc - 1))
            deciles.append(round(curve[idx], 4))
        record[f"{scorer_name}_gcc_deciles"] = deciles

        for ratio in FIXED_RATIOS:
            selected = fixed_summary(len(sentences), scores, ratio)
            variant = f"{scorer_name}_fixed{int(ratio*100)}"
            rouge = evaluate_summary(selected, sentences, reference_text)
            record[f"{variant}_rouge1_f"] = round(rouge["rouge1_f"], 5)
            record[f"{variant}_rouge2_f"] = round(rouge["rouge2_f"], 5)
            record[f"{variant}_rougeL_f"] = round(rouge["rougeL_f"], 5)

    return record

## Run the Experiment

Process `MAX_CNNDM` documents from the loaded data.

In [ ]:
import time

records = []
t0 = time.time()

examples_to_process = cnndm_examples[:MAX_CNNDM]

for i, ex in enumerate(examples_to_process):
    doc_id = ex.get("metadata_doc_id", str(i))
    # Use the article text stored in 'input' field
    article = ex.get("input", "")
    # Use the reference summary stored in 'output' field
    reference = ex.get("output", "")
    rec = process_document(article, reference, doc_id, "cnndm")
    if rec is not None:
        records.append(rec)
    print(f"Doc {i+1}/{len(examples_to_process)}: {doc_id} → {rec['n_sentences'] if rec else 'skipped'} sentences")

elapsed = time.time() - t0
print(f"\nProcessed {len(records)} docs in {elapsed:.2f}s")

## Aggregate Statistics

Compute mean ROUGE-1 F and compression ratio across all processed documents.

In [ ]:
def aggregate(records: list[dict]) -> dict:
    agg = {}
    for scorer_name in ["tf", "tfidf"]:
        agg[scorer_name] = {}
        for theta in THETAS:
            variant = f"{scorer_name}_theta{int(theta*10)}"
            sub = [r for r in records if f"{variant}_rouge1_f" in r]
            if not sub:
                continue
            agg[scorer_name][f"theta{int(theta*10)}"] = {
                "n": len(sub),
                "rouge1_f_mean": round(float(np.mean([r[f"{variant}_rouge1_f"] for r in sub])), 5),
                "rouge2_f_mean": round(float(np.mean([r[f"{variant}_rouge2_f"] for r in sub])), 5),
                "compression_ratio_mean": round(float(np.mean([r[f"{variant}_compression_ratio"] for r in sub])), 5),
                "ceiling_hit_frac": round(float(np.mean([r[f"{variant}_ceiling_hit"] for r in sub])), 4),
            }
        for ratio in FIXED_RATIOS:
            variant = f"{scorer_name}_fixed{int(ratio*100)}"
            sub = [r for r in records if f"{variant}_rouge1_f" in r]
            if not sub:
                continue
            agg[scorer_name][f"fixed{int(ratio*100)}"] = {
                "n": len(sub),
                "rouge1_f_mean": round(float(np.mean([r[f"{variant}_rouge1_f"] for r in sub])), 5),
                "rouge2_f_mean": round(float(np.mean([r[f"{variant}_rouge2_f"] for r in sub])), 5),
            }
    return agg

agg = aggregate(records)
print("Aggregates computed.")

## Results: ROUGE-1 F and Compression Ratio

Comparison of percolation threshold variants vs fixed-ratio baselines.

In [ ]:
print(f"{'Variant':<16} {'Scorer':<6} {'ROUGE-1 F':>10} {'ROUGE-2 F':>10} {'Comp.Ratio':>11} {'CeilHit':>8}")
print("-" * 65)

for scorer_name in ["tf", "tfidf"]:
    if scorer_name not in agg:
        continue
    for variant, stats in sorted(agg[scorer_name].items()):
        r1 = stats.get("rouge1_f_mean", float("nan"))
        r2 = stats.get("rouge2_f_mean", float("nan"))
        cr = stats.get("compression_ratio_mean", float("nan"))
        ch = stats.get("ceiling_hit_frac", float("nan"))
        cr_str = f"{cr:.3f}" if cr == cr else "n/a"
        ch_str = f"{ch:.3f}" if ch == ch else "n/a"
        print(f"{variant:<16} {scorer_name:<6} {r1:>10.4f} {r2:>10.4f} {cr_str:>11} {ch_str:>8}")
    print()

## Visualization: GCC Growth Curves and ROUGE-1 vs Compression Ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GCC growth curves for each document
ax = axes[0]
colors = {"tf": "steelblue", "tfidf": "darkorange"}
for rec in records:
    for scorer_name, color in colors.items():
        deciles = rec.get(f"{scorer_name}_gcc_deciles", [])
        if deciles:
            x = [0.1 * (i+1) for i in range(len(deciles))]
            ax.plot(x, deciles, color=color, alpha=0.5, linewidth=1.5,
                    label=scorer_name if rec == records[0] else "")
for t in THETAS:
    ax.axhline(t, linestyle="--", color="gray", linewidth=0.8, alpha=0.7)
    ax.text(1.01, t, f"θ={t}", va="center", fontsize=8, color="gray")
ax.set_xlabel("Fraction of sentences added")
ax.set_ylabel("GCC fraction of full doc")
ax.set_title("GCC Growth Curves (TF vs TF-IDF)")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

# Right: ROUGE-1 F vs compression ratio for percolation variants
ax2 = axes[1]
for scorer_name, color in colors.items():
    if scorer_name not in agg:
        continue
    xs, ys, labels = [], [], []
    for variant, stats in sorted(agg[scorer_name].items()):
        if "theta" not in variant:
            continue
        cr = stats.get("compression_ratio_mean")
        r1 = stats.get("rouge1_f_mean")
        if cr is not None and r1 is not None:
            xs.append(cr)
            ys.append(r1)
            labels.append(variant)
    ax2.plot(xs, ys, marker="o", color=color, label=scorer_name, linewidth=1.5)
    for x, y, lab in zip(xs, ys, labels):
        ax2.annotate(lab, (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)

# Add fixed-ratio baselines as horizontal dashed lines
for scorer_name, color in colors.items():
    if scorer_name not in agg:
        continue
    for ratio in FIXED_RATIOS:
        key = f"fixed{int(ratio*100)}"
        if key in agg[scorer_name]:
            r1 = agg[scorer_name][key].get("rouge1_f_mean")
            if r1 is not None:
                ax2.axhline(r1, linestyle=":", color=color, alpha=0.5, linewidth=1)

ax2.set_xlabel("Mean compression ratio (k*/n)")
ax2.set_ylabel("Mean ROUGE-1 F")
ax2.set_title("ROUGE-1 F vs Compression Ratio\n(dotted = fixed-ratio baselines)")
ax2.legend()

plt.tight_layout()
plt.savefig("percolation_summary_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to percolation_summary_results.png")